In [1]:
# Validation for Dual-Stream BERT checkpoint
# Imports and paths
import os, sys, torch
from datetime import datetime

# Ensure project root on path
project_root = "D:/BBNC/PICO/code/Airscope_toolbox/Software/Neuron_BERT"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Checkpoint and data dirs
checkpoint_path = "runs/two_stream_transformer_tube_test/trial_level_emb256_d4_mlp4_drop30_lr0.001_20251112-154352/best_model.pth"

assert os.path.exists(checkpoint_path), f"Checkpoint not found: {checkpoint_path}"
print("Using checkpoint:", checkpoint_path)


Using checkpoint: runs/two_stream_transformer_tube_test/trial_level_emb256_d4_mlp4_drop30_lr0.001_20251112-154352/best_model.pth


In [4]:
# Build datasets and dataloaders
from datasets import CalciumDataset_two_stream_pair_pca
from torch.utils.data import DataLoader

seq_length = 196
pca_dim = 128  # must match training PCA dim
seed = 42
val_cache_path = os.path.join(project_root, 'cache', f'val_data_seed_{seed}_pca.pkl')

val_dataset = CalciumDataset_two_stream_pair_pca(
    None,
    seq_length=seq_length,
    pca_dim=pca_dim,
    save_path=val_cache_path,
    training=False,
    random_crop=False,
    channel_shuffle_prob=0.0,
    channel_dropout_prob=0.0,
    time_mask_prob=0.0,
)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
print("Validation samples:", len(val_dataset))

Loading preprocessed data from D:/BBNC/PICO/code/Airscope_toolbox/Software/Neuron_BERT\cache\val_data_seed_42_pca.pkl...
Validation samples: 17


In [5]:
# Build model and load checkpoint
from model import DualStreamBERT
import torch

# Hyperparameters used in training (inferred from run folder)
model_kwargs = dict(
    input_dim=128,         # PCA components
    seq_length=196,
    embed_dim=256,
    depth=4,
    num_heads=4,
    mlp_ratio=4.0,
    num_classes=2,
    drop_ratio=0.3,
)

model = DualStreamBERT(**model_kwargs)
state = torch.load(checkpoint_path, map_location='cpu')
missing, unexpected = model.load_state_dict(state, strict=False)
print("Loaded state_dict. Missing:", missing, "Unexpected:", unexpected)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print("Device:", device)


Loaded state_dict. Missing: [] Unexpected: []
Device: cuda


In [6]:
# Run validation and compute metrics
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

criterion = nn.CrossEntropyLoss()
all_preds, all_labels = [], []
running_loss = 0.0

with torch.no_grad():
    for batch in val_loader:
        # Each batch returns (m1, m2, m1_mask+m2_mask combined?), but in two-stream it's (m1, m2, m1_mask, m2_mask, labels)
        if len(batch) == 5:
            m1, m2, m1_mask, m2_mask, labels = batch
        else:
            raise RuntimeError(f"Unexpected val batch structure: {type(batch)} with length {len(batch)}")
        m1, m2, m1_mask, m2_mask, labels = [x.to(device) for x in (m1, m2, m1_mask, m2_mask, labels)]

        outputs = model(m1, m2, m1_mask, m2_mask)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

val_loss = running_loss / len(all_labels)
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='binary', zero_division=0)
rec = recall_score(all_labels, all_preds, average='binary', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Accuracy: {acc * 100:.2f}% | Precision: {prec * 100:.2f}% | Recall: {rec * 100:.2f}% | F1: {f1 * 100:.2f}%")

# Save confusion matrix to the run folder
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['M2 Wins','M1 Wins'], yticklabels=['M2 Wins','M1 Wins'])
plt.title('Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
run_dir = os.path.dirname(checkpoint_path)
fig_path = os.path.join(run_dir, 'validation_confusion_matrix.pdf')
plt.tight_layout(); plt.savefig(fig_path); plt.close()
print('Saved confusion matrix to:', fig_path)


Validation Loss: 0.7059
Accuracy: 82.35% | Precision: 75.00% | Recall: 100.00% | F1: 85.71%
Confusion Matrix:
 [[5 3]
 [0 9]]
Saved confusion matrix to: runs/two_stream_transformer_tube_test/trial_level_emb256_d4_mlp4_drop30_lr0.001_20251112-154352\validation_confusion_matrix.pdf
Saved confusion matrix to: runs/two_stream_transformer_tube_test/trial_level_emb256_d4_mlp4_drop30_lr0.001_20251112-154352\validation_confusion_matrix.pdf
